In [2]:
!pip install git+https://github.com/openai/whisper.git
!pip install datasets==2.19.0 soundfile jiwer pyctcdecode

import os
import re
import random
import torch
import torch.nn as nn
import torch.nn.utils.rnn as rnn_utils
import numpy as np
import librosa
import whisper
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from jiwer import wer
from pyctcdecode import build_ctcdecoder
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device activated: {device}")

print("Loading Whisper model")
whisper_model = whisper.load_model("small").to(device)

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-50v6w4al
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-50v6w4al
  Resolved https://github.com/openai/whisper.git to commit cba3768142a28276a90f14907e4900372c0c3ee0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Device activated: cuda
Loading Whisper model


In [3]:
def buckwalter_to_arabic(text):
    bw_map = {
        "'": "ء", "|": "آ", ">": "أ", "&": "ؤ", "<": "إ", "}": "ئ",
        "A": "ا", "b": "ب", "p": "ة", "t": "ت", "v": "ث", "j": "ج",
        "H": "ح", "x": "خ", "d": "د", "*": "ذ", "r": "ر", "z": "ز",
        "s": "س", "$": "ش", "S": "ص", "D": "ض", "T": "ط", "Z": "ظ",
        "E": "ع", "g": "غ", "f": "ف", "q": "ق", "k": "ك", "l": "ل",
        "m": "م", "n": "ن", "h": "ه", "w": "و", "y": "ي",
        "F": "ً", "N": "ٌ", "K": "ٍ", "a": "َ", "u": "ُ", "i": "ِ",
        "~": "ّ", "o": "ْ", "-": "", "_": ""
    }
    return "".join([bw_map.get(c, c) for c in text])

def normalize_arabic_text(text):
    diacritics = re.compile("""ّ|َ|ً|ُ|ٌ|ِ|ٍ|ْ|ـ""", re.VERBOSE)
    text = re.sub(diacritics, '', text)
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ة", "ه", text)
    return text.strip()

def get_wer(ref, hyp):
    return wer(ref, hyp)

arabic_alphabet = "ابتثجحخدذرزسشصضطظعغفقكلمنهوي "
char_map = {char: i for i, char in enumerate(arabic_alphabet)}
char_map['<PAD>'] = len(char_map)
blank_index = len(char_map)
char_map['<BLANK>'] = blank_index

print("Alphabet dictionary and cleaning functions are ready")

Alphabet dictionary and cleaning functions are ready


In [4]:
!wget -c -q http://www.arabicspeechcorpus.com/arabic-speech-corpus.zip
!unzip -oq arabic-speech-corpus.zip

class ArabicSpeechDataset(Dataset):
    def __init__(self, corpus_dir, char_map):
        self.wav_dir = os.path.join(corpus_dir, "wav")
        self.char_map = char_map
        self.transcript_dict = {}
        transcript_path = os.path.join(corpus_dir, "orthographic-transcript.txt")

        if os.path.exists(transcript_path):
            with open(transcript_path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split('" "')
                    if len(parts) == 2:
                        wav_name = parts[0].replace('"', '').split('/')[-1]
                        text_bw = parts[1].replace('"', '')
                        self.transcript_dict[wav_name] = text_bw
        self.wav_files = [f for f in os.listdir(self.wav_dir) if f.endswith('.wav') and f in self.transcript_dict]

    def __len__(self): return len(self.wav_files)
    def __getitem__(self, idx):
        wav_file = self.wav_files[idx]
        wav_path = os.path.join(self.wav_dir, wav_file)
        y, sr = librosa.load(wav_path, sr=16000)
        spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
        spec_tensor = torch.FloatTensor(librosa.power_to_db(spec, ref=np.max))
        text_clean = normalize_arabic_text(buckwalter_to_arabic(self.transcript_dict[wav_file]))
        target = [self.char_map.get(c, self.char_map[' ']) for c in text_clean]
        return spec_tensor, target

fleurs_dataset = load_dataset("google/fleurs", "ar_eg", split="train", trust_remote_code=True)

class FleursDataset(Dataset):
    def __init__(self, hf_dataset, char_map):
        self.hf_dataset = hf_dataset
        self.char_map = char_map
    def __len__(self): return len(self.hf_dataset)
    def __getitem__(self, idx):
        sample = self.hf_dataset[idx]
        audio_array = sample['audio']['array']
        sr = sample['audio']['sampling_rate']
        if sr != 16000: audio_array = librosa.resample(y=audio_array, orig_sr=sr, target_sr=16000)
        spec = librosa.feature.melspectrogram(y=audio_array, sr=16000, n_mels=64)
        spec_tensor = torch.FloatTensor(librosa.power_to_db(spec, ref=np.max))
        text_clean = normalize_arabic_text(sample['transcription'])
        target = [self.char_map.get(c, self.char_map[' ']) for c in text_clean]
        return spec_tensor, target

def collate_fn(batch):
    specs, targets = zip(*batch)
    specs = [s.squeeze(0).permute(1, 0) for s in specs]

    MAX_SPECTROGRAM_TIME_STEPS = 308
    processed_specs = []
    for s in specs:
        if s.size(0) > MAX_SPECTROGRAM_TIME_STEPS:
            processed_specs.append(s[:MAX_SPECTROGRAM_TIME_STEPS, :])
        else:
            processed_specs.append(s)

    specs_padded = rnn_utils.pad_sequence(processed_specs, batch_first=True).permute(0, 2, 1).unsqueeze(1)
    targets = [torch.tensor(t) for t in targets]
    return specs_padded, targets

train_ds = ArabicSpeechDataset("arabic-speech-corpus", char_map)
fleurs_ds = FleursDataset(fleurs_dataset, char_map)
combined_ds = ConcatDataset([train_ds, fleurs_ds])
combined_loader = DataLoader(combined_ds, batch_size=4, shuffle=True, collate_fn=collate_fn)

print(f"Data combined Total: {len(combined_ds)} samples.")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Data combined Total: 3917 samples.


In [5]:
class ArabicAudioModel(nn.Module):
    def __init__(self, num_classes):
        super(ArabicAudioModel, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2)
        )

        self.lstm = nn.LSTM(
            input_size=1024, hidden_size=256, num_layers=1,
            batch_first=True, bidirectional=True
        )
        self.classifier = nn.Linear(256 * 2, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        sizes = x.size()
        x = x.view(sizes[0], sizes[1] * sizes[2], sizes[3]).transpose(1, 2)
        x, _ = self.lstm(x)
        return self.classifier(x)

print("successfully built the V1 architecture")

successfully built the V1 architecture


In [10]:
def train_robust(model, loader, optimizer, criterion, scheduler, device):
    model.train()
    total_loss = 0
    for i, (specs, targets) in enumerate(loader):
        specs = specs.to(device)


        batch_size, _, mels, time_steps = specs.size()
        augmented_specs = specs.clone()
        for b in range(batch_size):
            if time_steps > 20:
                t = random.randint(0, int(time_steps * 0.1))
                t0 = random.randint(0, time_steps - t)
                augmented_specs[b, 0, :, t0:t0+t] = 0
            f = random.randint(0, 10)
            f0 = random.randint(0, mels - f)
            augmented_specs[b, 0, f0:f0+f, :] = 0


        noise = torch.randn_like(augmented_specs) * 2.0
        augmented_specs = augmented_specs + noise

        # Corrected input_lengths to account for CNN downsampling
        input_lengths = torch.full(size=(augmented_specs.size(0),), fill_value=augmented_specs.size(3) // 4, dtype=torch.long)
        target_lengths = torch.LongTensor([len(t) for t in targets])
        targets_padded = rnn_utils.pad_sequence(targets, batch_first=True).to(device)

        optimizer.zero_grad()
        outputs = model(augmented_specs)
        outputs = torch.nn.functional.log_softmax(outputs, dim=2).permute(1, 0, 2)
        loss = criterion(outputs, targets_padded, input_lengths, target_lengths)

        if torch.isfinite(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    scheduler.step(avg_loss)
    return avg_loss, optimizer.param_groups[0]['lr']

print("Advanced training functions are ready")

Advanced training functions are ready


In [7]:

def decode_prediction(logits_tensor, rev_char_map, blank_idx):
    logits_np = logits_tensor.squeeze(0).cpu().numpy()
    pred_indices = np.argmax(logits_np, axis=1)

    decoded_text = []
    previous_idx = -1
    for idx in pred_indices:
        if idx != blank_idx and idx != previous_idx:
            decoded_text.append(rev_char_map[idx])
        previous_idx = idx
    return "".join(decoded_text)

print("Greedy Search tool is ready")

Greedy Search tool is ready


In [11]:
model_final = ArabicAudioModel(num_classes=len(char_map)).to(device)
criterion = nn.CTCLoss(blank=blank_index, zero_infinity=True)
optimizer = torch.optim.Adam(model_final.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)


epochs = 15

print(f"Starting deep model training (V1)")
for epoch in range(epochs):
    avg_l, current_lr = train_robust(model_final, combined_loader, optimizer, criterion, scheduler, device)
    print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_l:.4f} | LR: {current_lr:.6f}")

print("Training finished successfully")

Starting deep model training (V1)
Epoch 01/15 | Loss: 1.0985 | LR: 0.001000
Epoch 02/15 | Loss: 0.8960 | LR: 0.001000
Epoch 03/15 | Loss: 0.8495 | LR: 0.001000
Epoch 04/15 | Loss: 0.8153 | LR: 0.001000
Epoch 05/15 | Loss: 0.7915 | LR: 0.001000
Epoch 06/15 | Loss: 0.7812 | LR: 0.001000
Epoch 07/15 | Loss: 0.7549 | LR: 0.001000
Epoch 08/15 | Loss: 0.7502 | LR: 0.001000
Epoch 09/15 | Loss: 0.7351 | LR: 0.001000
Epoch 10/15 | Loss: 0.7166 | LR: 0.001000
Epoch 11/15 | Loss: 0.7228 | LR: 0.001000
Epoch 12/15 | Loss: 0.7047 | LR: 0.001000
Epoch 13/15 | Loss: 0.6934 | LR: 0.001000
Epoch 14/15 | Loss: 0.6893 | LR: 0.001000
Epoch 15/15 | Loss: 0.6754 | LR: 0.001000
Training finished successfully


In [16]:
sample_idx = random.randint(0, len(fleurs_dataset) - 1)
sample = fleurs_dataset[sample_idx]
true_text = sample['transcription']
audio_array = sample['audio']['array']
sr = sample['audio']['sampling_rate']

if sr != 16000: audio_array = librosa.resample(y=audio_array, orig_sr=sr, target_sr=16000)
true_text_clean = normalize_arabic_text(true_text)


model_final.eval()
spec = librosa.feature.melspectrogram(y=audio_array, sr=16000, n_mels=64)
my_tensor = torch.FloatTensor(librosa.power_to_db(spec, ref=np.max)).unsqueeze(0).unsqueeze(0).to(device)
with torch.no_grad():
    out = model_final(my_tensor)
    out = torch.nn.functional.log_softmax(out, dim=2)
    rev_map = {i: ch for ch, i in char_map.items()}
    my_pred_text = decode_prediction(out, rev_map, blank_index)


audio_float32 = audio_array.astype(np.float32)
result = whisper_model.transcribe(audio_float32, language="ar")
whisper_text_clean = normalize_arabic_text(result["text"])

print("\n" + "="*70)
print("Fair evaluation from dataset: Custom Model V1 vs Whisper")
print("="*70)
print(f"Original Text: {true_text_clean}")
print("-" * 70)
print(f" My Model Prediction: {my_pred_text}")
print(f"(WER): {get_wer(true_text_clean, my_pred_text if my_pred_text else ' '):.2f}")
print("-" * 70)
print(f" Whisper Prediction:: {whisper_text_clean}")
print(f"(WER): {get_wer(true_text_clean, whisper_text_clean if whisper_text_clean else ' '):.2f}")
print("="*70)


Fair evaluation from dataset: Custom Model V1 vs Whisper
Original Text: التسمم الداخلي قد لا تظهر عوارضه فورا التشخيص الفوري لا يمكن بناؤه عند ظهور اعراض كالتقيؤ والتي هي عامه بشكل كاف
----------------------------------------------------------------------
 My Model Prediction: لا ا ا ا الا ا ا ا ا ا ا ا ا ا ا  
(WER): 0.95
----------------------------------------------------------------------
 Whisper Prediction:: التسموم الداخلي قد لا تظهر عوارده فورا التشخيص الفوري لا يمكن بناءه عند زهور اعراض قلتقيق والتي هي عامه بشكل كافن
(WER): 0.29
